# 🎓 第7章 第3课：风格分析与归因

**本节目标**：

1. 用 **Sharpe 风格回归** 反推一个组合/基金暴露在哪些因子上；
2. 做 **风险归因**：组合波动里，多少来自因子、多少来自特质；
3. 做 **业绩归因**：组合收益里，多少来自因子暴露、多少是 α。

对应原书第7章风格分析与风险归因部分。

In [1]:
import sys
from pathlib import Path

def _find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'src' / 'factor_zero').exists():
            return p
    raise FileNotFoundError('找不到仓库根目录，请从仓库内启动 notebook')
ROOT = _find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from factor_zero.data import load_ashare_panel
from factor_zero.factors import build_factor_returns
from factor_zero.anomalies import rolling_ivol
from factor_zero.portfolio import (
    FACTOR_ORDER, build_style_exposures, factor_covariance,
    specific_variance, portfolio_risk, min_variance_weights,
    max_sharpe_weights, style_regression, smart_beta_weights,
)
from factor_zero.statistics import factor_summary, nw_t_stat

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
print('✅ 环境就绪，开始学习！')

✅ 环境就绪，开始学习！


## 1) Sharpe 风格回归：反推组合的因子暴露

构造一个“假装是基金”的组合：

```
fund = 0.6·MKT + 0.2·HML + 0.1·RMW + 0.1·WML + α + 噪声
```

然后只用基金收益和因子收益做回归，看能不能把 0.6/0.2/0.1/0.1 反推出来。

In [2]:
panel = load_ashare_panel()
factors = build_factor_returns(panel, n_groups=10)
rng = np.random.default_rng(7)
true_beta = pd.Series({'MKT': 0.6, 'SMB': 0.0, 'HML': 0.2, 'WML': 0.1,
                       'RMW': 0.1, 'CMA': 0.0, 'TURN': 0.0})
alpha_true = 0.002
fund = factors[FACTOR_ORDER] @ true_beta + alpha_true + rng.normal(0, 0.005, len(factors))
alpha, betas, r2, resid = style_regression(fund.dropna(), factors[FACTOR_ORDER])
print(f'回归 α（月）: {alpha*100:.3f}%   R²: {r2:.3f}')
pd.DataFrame({'真实暴露': true_beta, '回归估计': betas}).round(3)

回归 α（月）: 0.122%   R²: 0.982


,真实暴露,回归估计
MKT,0.6,0.585
SMB,0.0,0.000
HML,0.2,0.202
WML,0.1,0.104
RMW,0.1,0.099
CMA,0.0,-0.019
TURN,0.0,0.004


## 2) 风险归因：价值倾斜组合的风险从哪里来？

用第 1 课的迷你风险模型，对“高 BM 前 20% 等权组合”做 Euler 分解。

In [3]:
ret = panel['ret']
beta = ret.rolling(24).cov(panel['market']).div(panel['market'].rolling(24).var(), axis=0)
ivol = rolling_ivol(ret, factors[FACTOR_ORDER], window=36)
date = ret.index[-1]
B = build_style_exposures(panel, date, beta)
fcov = factor_covariance(factors, date, window=36)
sv = specific_variance(ivol, date, B.index)
top = B['HML'].sort_values(ascending=False).head(max(10, int(len(B) * 0.2))).index
w_tilt = pd.Series(1.0 / len(top), index=top)
risk = portfolio_risk(w_tilt, B, fcov, sv)
contrib = risk['factor_contrib'].copy(); contrib['特质'] = risk['specific_contrib']
print(f'总波动: {risk["total_vol"]*100:.2f}%  因子: {risk["factor_vol"]*100:.2f}%  特质: {risk["specific_vol"]*100:.2f}%')
(contrib * 100).round(2)

总波动: 50.12%  因子: 49.91%  特质: 4.59%


MKT      1.50
SMB     -0.51
HML     42.54
WML      1.18
RMW      4.19
CMA     -0.26
TURN     1.06
特质       0.42
dtype: float64

## 3) 业绩归因：收益里多少来自因子？

对**价值倾斜组合**（高 BM 前 20%，等权）做归因：
用“组合暴露 × 因子收益”作为因子贡献，剩余部分作为 α。
看最近 36 个月的年化贡献。

In [4]:
window = factors.index[-36:]
port_ret = (ret.loc[window, top] * w_tilt).sum(axis=1)
Bp = w_tilt.reindex(B.index).fillna(0.0) @ B
factor_contrib = factors.loc[window, FACTOR_ORDER].mul(Bp, axis=1)
alpha_series = port_ret - factor_contrib.sum(axis=1)
summary = pd.DataFrame({
    '年化贡献': pd.concat([
        factor_contrib.mean() * 12,
        pd.Series({'α': alpha_series.mean() * 12}),
    ]),
})
summary.round(4)

,年化贡献
MKT,-0.0185
SMB,-0.0080
HML,-0.1643
WML,0.0232
RMW,-0.1197
CMA,0.0043
TURN,0.0333
α,0.3198


> 🔍 **本例 α 很大（年化约 32%）**：这更可能是模型错位/数据面板问题，而不是真实选股能力。
> 归因结果高度依赖风险模型、因子定义和样本；不要看到一个漂亮 α 就当真。


## ✋ 试一试

1. 把 `true_beta` 的 HML 改成 0.4、RMW 改成 −0.1，回归能反推出来吗？
2. 把噪声从 0.005 改成 0.02，R² 和系数估计怎么变？
3. 把价值倾斜组合换成等权/最小方差组合，风险/业绩归因怎么变？

## ✅ 本节小结

- 风格回归用因子收益反推组合暴露；
- 风险归因把波动拆成因子贡献 + 特质贡献；
- 业绩归因把收益拆成因子贡献 + α；
- 归因是“解释”，不是“预测”；模型不同，归因结果也会不同。

> 下一课：结业项目——把前面所有工具串起来。